# Entrenamiento del clasificador de radiografías (NORMAL / NEUMONIA + detección de "otra cosa")

Este notebook reemplaza el modelo YOLO de clasificación de neumonía (`modelo_neumonia.pt`) por un modelo de
**transfer learning con PyTorch/torchvision**, usando por defecto **InceptionV3** (recomendado por el profesor).

El modelo YOLO de verificación de tórax (`modelo_torax.pt`) **se mantiene igual** y no se toca en este notebook.

## Enfoque: solo entrenamos 2 clases, no 3

No hace falta conseguir ni subir imágenes de "otra enfermedad". El modelo se entrena **únicamente** con las
imágenes que ya tienes: `NORMAL` y `NEUMONIA`.

La tercera categoría (`OTRA_ENFERMEDAD`) no es una clase que el modelo aprenda: se obtiene por
**umbral de confianza**. Como el modelo hace `softmax` entre 2 clases, siempre va a "elegir" la que le
parezca más probable, incluso si la imagen es de otra cosa completamente distinta (un tumor, una fractura,
una imagen borrosa, etc.). La idea es:

- Si el modelo está **muy seguro** (confianza alta) de que es `NORMAL` o `NEUMONIA`, se reporta esa clase.
- Si el modelo **no está seguro** (confianza baja, cercana a 50%), en vez de forzar una respuesta se reporta
  `OTRA_ENFERMEDAD` ("no es claramente sano ni neumonía, esto debería revisarse").

Al final obtendrás dos archivos para copiar a `agente-AI/clasificador/ml_models/`:

- `modelo_neumonia.pt` (pesos del modelo entrenado, solo 2 clases: NORMAL/NEUMONIA)
- `clases_neumonia.json` (clases + arquitectura + **umbral de confianza** elegido en este notebook)


## 1. Preparar el entorno

Colab ya trae PyTorch y torchvision instalados. Solo instalamos un par de utilidades extra.

In [ ]:
!pip install -q split-folders scikit-learn grad-cam


## 2. Montar Google Drive

Vamos a guardar el dataset y los resultados en tu Drive para no perderlos si se reinicia el entorno de Colab.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 3. Estructura de carpetas esperada

Sube tu dataset (el que ya tienes) a Google Drive respetando esta estructura (los nombres de las carpetas
de clase deben ser exactamente estos, en mayúsculas):

```
/content/drive/MyDrive/dataset_pulmones_raw/
    NORMAL/            <- tus radiografías normales
    NEUMONIA/          <- tus radiografías con neumonía
```

No se necesita ninguna carpeta ni dataset extra para "otra enfermedad": esa categoría se calcula después,
con el umbral de confianza (ver sección 12).

## 4. Dividir en train / val / test (80% / 10% / 10%)

`splitfolders` toma tu carpeta con las 2 subcarpetas de clases y genera automáticamente la división,
manteniendo la misma proporción de clases en cada conjunto.

In [ ]:
import splitfolders

RUTA_DATASET_RAW = "/content/drive/MyDrive/dataset_pulmones_raw"
RUTA_DATASET_SPLIT = "/content/drive/MyDrive/dataset_pulmones_split"

splitfolders.ratio(
    RUTA_DATASET_RAW,
    output=RUTA_DATASET_SPLIT,
    seed=42,
    ratio=(0.8, 0.1, 0.1),
)


## 5. Elegir el modelo base

Puedes cambiar `MODEL_NAME` por `"vgg16"`, `"mobilenet_v3_large"` o `"resnet50"` sin tener que tocar el resto
del notebook. Por defecto usamos `inception_v3`, tal como pidió el profesor.

In [ ]:
MODEL_NAME = "inception_v3"  # opciones: "inception_v3", "vgg16", "mobilenet_v3_large", "resnet50"

# InceptionV3 exige imágenes de 299x299. Los demás modelos usan 224x224.
IMG_SIZE = 299 if MODEL_NAME == "inception_v3" else 224
BATCH_SIZE = 32


## 6. Preprocesamiento, normalización de contraste y transforms

Usamos la media/desviación estándar de ImageNet porque los modelos vienen preentrenados en ese dataset.

Varios detalles para que el modelo generalice a radiografías de **otras fuentes** (otros hospitales, equipos,
o incluso capturas de pantalla de un visor DICOM, no solo las de tu dataset de entrenamiento):

- **Recorte automático de bordes negros**: muchas radiografías "reales" (capturas de un visor médico) traen
  barras negras a los lados con texto de metadatos, reglas, logos, etc. En vez de recortar un porcentaje fijo,
  detectamos automáticamente qué parte de la imagen tiene contenido real (no es una barra negra con un poco
  de texto) y recortamos a esa región. En imágenes ya limpias (como probablemente las tuyas) esto casi no
  cambia nada.
- **Normalización de contraste (CLAHE)**: aplicamos ecualización de histograma adaptativa a cada imagen antes
  de meterla al modelo. Esto es clave: si tu dataset viene de una fuente con un contraste/brillo particular
  (muy probable si descargaste un dataset de Kaggle), el modelo puede "hacer trampa" aprendiendo a reconocer
  ese estilo de imagen en vez del patrón médico real. CLAHE normaliza el contraste de forma pareja en
  cualquier radiografía, sin importar su fuente, quitándole esa "trampa" al modelo.
- `Resize` + `CenterCrop`/`RandomCrop` (esquema estándar de ImageNet) y `RandomErasing` en entrenamiento, como
  ya teníamos, para mayor robustez adicional.

In [ ]:
import random

import cv2
import numpy as np
import torch
from PIL import Image
from torchvision import datasets, transforms
import torchvision.transforms.functional as TF
from torch.utils.data import DataLoader

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# Esquema estándar de ImageNet: resize a ~1/0.875 del tamaño final, luego crop al tamaño final.
RESIZE_SIZE = int(round(IMG_SIZE / 0.875))


def recortar_bordes_negros(imagen, umbral_intensidad=15, fraccion_minima=0.15):
    """Recorta barras/márgenes negros (con texto de metadatos, reglas, etc.)
    conservando solo la región donde la mayoría de los píxeles tienen contenido real.
    """
    gris = np.array(imagen.convert("L"), dtype=np.float32)

    fraccion_por_columna = (gris > umbral_intensidad).mean(axis=0)
    fraccion_por_fila = (gris > umbral_intensidad).mean(axis=1)

    columnas_validas = np.where(fraccion_por_columna > fraccion_minima)[0]
    filas_validas = np.where(fraccion_por_fila > fraccion_minima)[0]

    if columnas_validas.size == 0 or filas_validas.size == 0:
        return imagen

    izquierda, derecha = int(columnas_validas[0]), int(columnas_validas[-1])
    arriba, abajo = int(filas_validas[0]), int(filas_validas[-1])

    return imagen.crop((izquierda, arriba, derecha + 1, abajo + 1))


def rotar_con_relleno(imagen, grados_max=10):
    """Rota la imagen un ángulo aleatorio y rellena las esquinas vacías con el tono
    promedio de la propia imagen, en vez del negro puro por defecto. Un relleno negro
    generaría una cuña con borde recto muy marcado, justo el tipo de "borde artificial"
    que `recortar_bordes_negros` le enseña al modelo a ignorar: sin este cambio, el
    augmentation estaría reintroduciendo el mismo patrón que tratamos de eliminar.
    """
    angulo = random.uniform(-grados_max, grados_max)
    color_relleno = tuple(int(c) for c in np.array(imagen).reshape(-1, 3).mean(axis=0))
    return TF.rotate(imagen, angulo, fill=color_relleno)


def normalizar_contraste(imagen):
    """Ecualización de histograma adaptativa (CLAHE), para que el modelo no dependa
    del contraste/brillo específico de la fuente de cada radiografía.
    """
    gris = np.array(imagen.convert("L"))
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    ecualizada = clahe.apply(gris)
    return Image.fromarray(ecualizada).convert("RGB")


transform_train = transforms.Compose([
    transforms.Lambda(recortar_bordes_negros),
    transforms.Lambda(normalizar_contraste),
    transforms.Resize((RESIZE_SIZE, RESIZE_SIZE)),
    transforms.RandomCrop((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.Lambda(rotar_con_relleno),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0))], p=0.25),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    transforms.RandomErasing(p=0.35, scale=(0.02, 0.15), ratio=(0.3, 3.3)),
])

transform_eval = transforms.Compose([
    transforms.Lambda(recortar_bordes_negros),
    transforms.Lambda(normalizar_contraste),
    transforms.Resize((RESIZE_SIZE, RESIZE_SIZE)),
    transforms.CenterCrop((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

dataset_train = datasets.ImageFolder(f"{RUTA_DATASET_SPLIT}/train", transform=transform_train)
dataset_val = datasets.ImageFolder(f"{RUTA_DATASET_SPLIT}/val", transform=transform_eval)
dataset_test = datasets.ImageFolder(f"{RUTA_DATASET_SPLIT}/test", transform=transform_eval)

loader_train = DataLoader(dataset_train, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
loader_val = DataLoader(dataset_val, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
loader_test = DataLoader(dataset_test, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

CLASES = dataset_train.classes
print("Orden de clases detectado (IMPORTANTE, debe coincidir en Django):", CLASES)
assert len(CLASES) == 2, "Se esperaban exactamente 2 clases: NORMAL y NEUMONIA"


## 6.1. Comparar la foto original contra la foto ya preprocesada

Antes de entrenar, conviene ver con los propios ojos qué le hace el pipeline de preprocesamiento
(recorte de bordes negros, CLAHE, resize/crop y augmentation) a una radiografía real. Esta celda toma
una imagen al azar del set de entrenamiento y muestra lado a lado:

- **Izquierda**: la imagen tal cual se subió, sin ningún parámetro aplicado.
- **Derecha**: la misma imagen después de aplicarle el pipeline (`transform_train`, sin contar
  `ToTensor`/`Normalize`, que no se pueden mostrar como imagen).

Puedes volver a correr esta celda las veces que quieras para ver otros ejemplos (el recorte/CLAHE es
siempre igual, pero el augmentation cambia cada vez porque es aleatorio).

In [ ]:
import random
import matplotlib.pyplot as plt

# Mismo pipeline que transform_train, pero sin ToTensor/Normalize/RandomErasing para poder
# mostrar el resultado como imagen normal.
transform_train_visual = transforms.Compose([
    transforms.Lambda(recortar_bordes_negros),
    transforms.Lambda(normalizar_contraste),
    transforms.Resize((RESIZE_SIZE, RESIZE_SIZE)),
    transforms.RandomCrop((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.Lambda(rotar_con_relleno),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0))], p=0.25),
])

ruta_ejemplo, _ = random.choice(dataset_train.samples)
imagen_original = Image.open(ruta_ejemplo).convert("RGB")
imagen_procesada = transform_train_visual(imagen_original)

figura, ejes = plt.subplots(1, 2, figsize=(10, 5))
ejes[0].imshow(imagen_original)
ejes[0].set_title("Original (sin nada aplicado)")
ejes[0].axis("off")
ejes[1].imshow(imagen_procesada)
ejes[1].set_title("Con preprocesamiento + augmentation aplicados")
ejes[1].axis("off")
plt.tight_layout()
plt.show()

## 7. Construir el modelo (transfer learning)

Cargamos los pesos preentrenados en ImageNet y reemplazamos la última capa por una con 2 salidas
(NORMAL / NEUMONIA). En la Fase 1 congelamos el resto del modelo para entrenar solo esa capa nueva.

In [ ]:
import torch.nn as nn
from torchvision import models

def crear_modelo(model_name, num_clases):
    if model_name == "inception_v3":
        modelo = models.inception_v3(weights=models.Inception_V3_Weights.IMAGENET1K_V1)
        modelo.fc = nn.Linear(modelo.fc.in_features, num_clases)
        modelo.AuxLogits.fc = nn.Linear(modelo.AuxLogits.fc.in_features, num_clases)

    elif model_name == "vgg16":
        modelo = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
        modelo.classifier[6] = nn.Linear(modelo.classifier[6].in_features, num_clases)

    elif model_name == "mobilenet_v3_large":
        modelo = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V1)
        modelo.classifier[3] = nn.Linear(modelo.classifier[3].in_features, num_clases)

    elif model_name == "resnet50":
        modelo = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        modelo.fc = nn.Linear(modelo.fc.in_features, num_clases)

    else:
        raise ValueError(f"Modelo no soportado: {model_name}")

    return modelo


CAPAS_FASE1 = {
    "inception_v3": ("fc", "AuxLogits.fc"),
    "vgg16": ("classifier",),
    "mobilenet_v3_large": ("classifier",),
    "resnet50": ("fc",),
}

# Solo el último bloque convolucional de cada arquitectura, además de la capa final.
# Dejar el resto de la base congelada reduce el riesgo de sobreajustar a detalles
# específicos de tu dataset (formato, contraste, recorte) cuando el dataset es chico.
CAPAS_FASE2 = {
    "inception_v3": ("fc", "AuxLogits.fc", "Mixed_7"),
    "vgg16": ("classifier", "features.24", "features.26", "features.28"),
    "mobilenet_v3_large": ("classifier", "features.15", "features.16"),
    "resnet50": ("fc", "layer4"),
}


def congelar_base(modelo, model_name):
    prefijos = CAPAS_FASE1[model_name]
    for nombre, parametro in modelo.named_parameters():
        parametro.requires_grad = any(nombre.startswith(prefijo) for prefijo in prefijos)


def descongelar_ultimas_capas(modelo, model_name):
    prefijos = CAPAS_FASE2[model_name]
    for nombre, parametro in modelo.named_parameters():
        parametro.requires_grad = any(nombre.startswith(prefijo) for prefijo in prefijos)


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Usando dispositivo:", DEVICE)

modelo = crear_modelo(MODEL_NAME, num_clases=len(CLASES))
congelar_base(modelo, MODEL_NAME)
modelo = modelo.to(DEVICE)


## 8. Pesos de clase (por si NORMAL/NEUMONIA quedan desbalanceadas)

Si una clase tiene muchas más imágenes que la otra, esto evita que el modelo se sesgue hacia la mayoritaria.

In [ ]:
from collections import Counter

conteo = Counter(dataset_train.targets)
total = sum(conteo.values())
pesos_clase = [total / (len(conteo) * conteo[i]) for i in range(len(CLASES))]
pesos_clase = torch.tensor(pesos_clase, dtype=torch.float32).to(DEVICE)

print("Conteo por clase:", {CLASES[i]: conteo[i] for i in conteo})
print("Pesos usados en la pérdida:", pesos_clase.tolist())

criterio = nn.CrossEntropyLoss(weight=pesos_clase)


## 9. Función de entrenamiento

Entrena y valida por época, y se queda con los pesos de la época con mejor accuracy de validación.
Maneja el caso especial de InceptionV3, que durante el entrenamiento devuelve dos salidas (la principal
y la auxiliar).

In [ ]:
import copy
import time


def entrenar(modelo, epocas, tasa_aprendizaje):
    optimizador = torch.optim.Adam(
        filter(lambda p: p.requires_grad, modelo.parameters()),
        lr=tasa_aprendizaje,
        weight_decay=1e-4,
    )

    mejor_pesos = copy.deepcopy(modelo.state_dict())
    mejor_accuracy = 0.0

    for epoca in range(epocas):
        inicio = time.time()

        # --- entrenamiento ---
        modelo.train()
        perdida_total = 0.0

        for imagenes, etiquetas in loader_train:
            imagenes, etiquetas = imagenes.to(DEVICE), etiquetas.to(DEVICE)

            optimizador.zero_grad()
            salida = modelo(imagenes)

            if MODEL_NAME == "inception_v3" and isinstance(salida, tuple):
                logits, aux_logits = salida
                perdida = criterio(logits, etiquetas) + 0.4 * criterio(aux_logits, etiquetas)
            else:
                perdida = criterio(salida, etiquetas)

            perdida.backward()
            optimizador.step()

            perdida_total += perdida.item() * imagenes.size(0)

        perdida_epoca = perdida_total / len(dataset_train)

        # --- validación ---
        modelo.eval()
        correctos = 0

        with torch.no_grad():
            for imagenes, etiquetas in loader_val:
                imagenes, etiquetas = imagenes.to(DEVICE), etiquetas.to(DEVICE)
                salida = modelo(imagenes)
                predicciones = torch.argmax(salida, dim=1)
                correctos += (predicciones == etiquetas).sum().item()

        accuracy_val = correctos / len(dataset_val)
        duracion = time.time() - inicio

        print(
            f"Época {epoca + 1}/{epocas} - "
            f"pérdida train: {perdida_epoca:.4f} - "
            f"accuracy val: {accuracy_val:.4f} - "
            f"{duracion:.1f}s"
        )

        if accuracy_val > mejor_accuracy:
            mejor_accuracy = accuracy_val
            mejor_pesos = copy.deepcopy(modelo.state_dict())

    modelo.load_state_dict(mejor_pesos)
    print(f"Mejor accuracy de validación: {mejor_accuracy:.4f}")
    return modelo


## 10. Fase 1: entrenar solo la capa nueva

Con la base congelada, entrenamos pocas épocas para que la capa final aprenda a distinguir NORMAL/NEUMONIA
sobre las características que ya "sabe ver" el modelo preentrenado.

In [ ]:
modelo = entrenar(modelo, epocas=8, tasa_aprendizaje=1e-3)


## 11. Fase 2: fine-tuning de las últimas capas (no de todo el modelo)

Descongelamos solo el último bloque convolucional (además de la capa final) y seguimos entrenando con una
tasa de aprendizaje baja. A propósito **no** descongelamos todo el modelo: con un dataset mediano/chico,
volver entrenable a un modelo tan grande completo tiende a sobreajustar a detalles específicos de tu fuente
de imágenes (contraste, recortes, formato) en vez de aprender el patrón real de neumonía, que es justo lo que
te pasó con la radiografía de otra clínica.

In [ ]:
descongelar_ultimas_capas(modelo, MODEL_NAME)
modelo = entrenar(modelo, epocas=6, tasa_aprendizaje=1e-4)


## 12. Evaluación en el conjunto de test

Matriz de confusión y reporte de precisión/recall/f1, sobre imágenes que el modelo nunca vio.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

modelo.eval()
todas_predicciones = []
todas_etiquetas = []

with torch.no_grad():
    for imagenes, etiquetas in loader_test:
        imagenes = imagenes.to(DEVICE)
        salida = modelo(imagenes)
        predicciones = torch.argmax(salida, dim=1).cpu().numpy()
        todas_predicciones.extend(predicciones)
        todas_etiquetas.extend(etiquetas.numpy())

print(classification_report(todas_etiquetas, todas_predicciones, target_names=CLASES))

matriz = confusion_matrix(todas_etiquetas, todas_predicciones)
plt.figure(figsize=(5, 4))
plt.imshow(matriz, cmap="Blues")
plt.title("Matriz de confusión")
plt.colorbar()
plt.xticks(range(len(CLASES)), CLASES, rotation=45)
plt.yticks(range(len(CLASES)), CLASES)
for i in range(len(CLASES)):
    for j in range(len(CLASES)):
        plt.text(j, i, matriz[i, j], ha="center", va="center")
plt.ylabel("Real")
plt.xlabel("Predicho")
plt.tight_layout()
plt.show()


## 12.1. Mapa de calor (Grad-CAM) sobre imágenes de test

Para confiar en el modelo (y no solo en el accuracy), conviene ver *en qué parte de la radiografía se fija*
para decidir NORMAL o NEUMONIA. Grad-CAM genera un mapa de calor a partir de los gradientes de la última
capa convolucional: las zonas más "calientes" (rojo/amarillo) son las que más influyeron en la predicción.

Si el modelo aprendió el patrón médico real, el mapa de calor debería concentrarse dentro de los pulmones
(sobre todo marcando las zonas de infiltrado en los casos de NEUMONIA). Si en cambio se enciende sobre
texto, bordes o zonas fuera del tórax, es señal de que el modelo está "haciendo trampa" con algún atajo
en vez de aprender el patrón real, y convendría revisar el dataset o el preprocesamiento.

Esta celda toma unas imágenes al azar del set de test y muestra, para cada una: la imagen preprocesada (lo
que realmente "ve" el modelo, sin augmentation porque es evaluación), el mapa de calor superpuesto, la
clase real y la clase predicha con su confianza. Puedes volver a correrla para ver otros ejemplos.

In [ ]:
import random

import matplotlib.pyplot as plt
import numpy as np
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

# Ultima capa convolucional de cada arquitectura, donde Grad-CAM calcula el mapa de calor.
CAPAS_GRADCAM = {
    "inception_v3": lambda m: [m.Mixed_7c],
    "vgg16": lambda m: [m.features[-1]],
    "mobilenet_v3_large": lambda m: [m.features[-1]],
    "resnet50": lambda m: [m.layer4[-1]],
}

# Mismo pipeline que transform_eval, pero sin ToTensor/Normalize, para poder mostrar la imagen
# preprocesada (la que realmente entra al modelo) como imagen normal.
transform_eval_visual = transforms.Compose([
    transforms.Lambda(recortar_bordes_negros),
    transforms.Lambda(normalizar_contraste),
    transforms.Resize((RESIZE_SIZE, RESIZE_SIZE)),
    transforms.CenterCrop((IMG_SIZE, IMG_SIZE)),
])

modelo.eval()
cam = GradCAM(model=modelo, target_layers=CAPAS_GRADCAM[MODEL_NAME](modelo))

N_POR_CLASE = 2

# Muestreamos por clase (no del test set completo al azar) para poder comparar directamente
# donde se enciende el mapa de calor en NORMAL vs NEUMONIA en la misma corrida.
muestras = []
for indice_clase in range(len(CLASES)):
    candidatos = [muestra for muestra in dataset_test.samples if muestra[1] == indice_clase]
    muestras.extend(random.sample(candidatos, min(N_POR_CLASE, len(candidatos))))
random.shuffle(muestras)

# Buscamos, en todo el test set, el caso donde el modelo estuvo MENOS seguro (probabilidad máxima
# más baja). No existe una clase real "OTRA_ENFERMEDAD" en el dataset (no se entrena como tal), pero
# este es el ejemplo más parecido: el tipo de imagen "dudosa" que el umbral de confianza (sección 13)
# terminaría marcando como OTRA_ENFERMEDAD en vez de forzar NORMAL o NEUMONIA. Lo agregamos siempre
# al final de la lista para poder identificarlo en el título.
confianzas_test = []
with torch.no_grad():
    for imagenes, _ in loader_test:
        imagenes = imagenes.to(DEVICE)
        probabilidades = torch.softmax(modelo(imagenes), dim=1)
        confianzas_test.extend(torch.max(probabilidades, dim=1).values.cpu().tolist())

indice_incierto = int(np.argmin(confianzas_test))
muestras.append(dataset_test.samples[indice_incierto])

figura, ejes = plt.subplots(1, len(muestras), figsize=(5 * len(muestras), 5))

for posicion, (eje, (ruta, etiqueta_real)) in enumerate(zip(ejes, muestras)):
    imagen_original = Image.open(ruta).convert("RGB")
    imagen_visual = transform_eval_visual(imagen_original)
    imagen_input = transform_eval(imagen_original).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        salida = modelo(imagen_input)
        probabilidades = torch.softmax(salida, dim=1)[0]
    indice_predicho = int(torch.argmax(probabilidades).item())
    confianza = float(probabilidades[indice_predicho].item()) * 100

    mapa_calor = cam(input_tensor=imagen_input, targets=[ClassifierOutputTarget(indice_predicho)])[0]
    imagen_flotante = np.array(imagen_visual).astype(np.float32) / 255.0
    visualizacion = show_cam_on_image(imagen_flotante, mapa_calor, use_rgb=True)

    eje.imshow(visualizacion)
    marca = "OK" if CLASES[etiqueta_real] == CLASES[indice_predicho] else "X"
    es_incierto = posicion == len(muestras) - 1
    sufijo = " [caso mas incierto ~ tipo OTRA_ENFERMEDAD]" if es_incierto else ""
    titulo = f"Real: {CLASES[etiqueta_real]} | Pred: {CLASES[indice_predicho]} ({confianza:.1f}%) {marca}{sufijo}"
    eje.set_title(titulo, fontsize=9)
    eje.axis("off")

plt.tight_layout()
plt.show()

## 13. Elegir el umbral de confianza para "OTRA_ENFERMEDAD"

Aquí está la parte clave de tu pedido: en vez de entrenar una tercera clase, revisamos qué tan "seguro" está
el modelo cuando acierta en el set de test, y elegimos un umbral de confianza por debajo del cual el sistema
va a reportar `OTRA_ENFERMEDAD` en vez de forzar NORMAL o NEUMONIA.

El histograma de abajo muestra la confianza máxima (`softmax`) que el modelo asigna a cada imagen de test.
Si casi todas las predicciones correctas tienen confianza alta (por ejemplo > 0.85), puedes usar un umbral
alto sin perder casos válidos. Si el modelo tiende a estar menos seguro, usa un umbral más bajo (por ejemplo
0.70) para no marcar como "otra cosa" casos que sí eran NORMAL o NEUMONIA reales.

In [ ]:
import numpy as np

modelo.eval()
confianzas_maximas = []

with torch.no_grad():
    for imagenes, etiquetas in loader_test:
        imagenes = imagenes.to(DEVICE)
        salida = modelo(imagenes)
        probabilidades = torch.softmax(salida, dim=1)
        confianzas_maximas.extend(torch.max(probabilidades, dim=1).values.cpu().numpy())

confianzas_maximas = np.array(confianzas_maximas)

plt.figure(figsize=(6, 4))
plt.hist(confianzas_maximas, bins=20, color="#7b2ff7", edgecolor="white")
plt.title("Distribución de confianza máxima (set de test)")
plt.xlabel("Confianza (softmax)")
plt.ylabel("Cantidad de imágenes")
plt.axvline(np.median(confianzas_maximas), color="black", linestyle="--", label="Mediana")
plt.legend()
plt.tight_layout()
plt.show()

print("Percentiles de confianza en test:")
for percentil in [5, 10, 25, 50]:
    valor = np.percentile(confianzas_maximas, percentil)
    print(f"  P{percentil}: {valor:.3f}")


In [ ]:
# Ajusta este valor según el histograma/percentiles de arriba.
# Ejemplo de criterio: usa el percentil 10 (o un poco menos) para que solo el 10% de las
# predicciones "normales" del modelo caigan por debajo del umbral.
# Un punto de partida razonable si no quieres analizar el histograma es 0.70-0.75.

UMBRAL_CONFIANZA = 0.75

print(f"Umbral de confianza elegido: {UMBRAL_CONFIANZA}")
print(
    f"Con este umbral, el {100 * (confianzas_maximas < UMBRAL_CONFIANZA).mean():.1f}% "
    "de las imágenes de test se habrían marcado como OTRA_ENFERMEDAD."
)


## 14. Guardar y descargar el modelo

Genera `modelo_neumonia.pt` (pesos) y `clases_neumonia.json` (clases + arquitectura + umbral de confianza).
Copia ambos archivos a `agente-AI/clasificador/ml_models/` en tu proyecto, reemplazando el
`modelo_neumonia.pt` viejo de YOLO.

In [ ]:
import json
from google.colab import files

torch.save(modelo.state_dict(), "modelo_neumonia.pt")

with open("clases_neumonia.json", "w", encoding="utf-8") as archivo:
    json.dump(
        {
            "model_name": MODEL_NAME,
            "clases": CLASES,
            "umbral_confianza": UMBRAL_CONFIANZA,
        },
        archivo,
        ensure_ascii=False,
        indent=2,
    )

files.download("modelo_neumonia.pt")
files.download("clases_neumonia.json")


## 15. Siguiente paso

En el proyecto Django (`agente-AI`), el archivo `clasificador/services/predictor.py` ya está preparado para:

1. Cargar este modelo de 2 clases con `torchvision` en lugar de YOLO.
2. Leer `clases_neumonia.json` para saber qué arquitectura reconstruir y qué clases usar.
3. Comparar la confianza máxima de la predicción contra `umbral_confianza`: si es menor, reporta
   `OTRA_ENFERMEDAD` en vez de `NORMAL`/`NEUMONIA`.

Si después de probar la app en varios casos reales ves que marca demasiadas veces "OTRA_ENFERMEDAD" (umbral
muy alto) o casi nunca la marca (umbral muy bajo), puedes simplemente editar el valor `umbral_confianza`
dentro de `clases_neumonia.json` sin tener que volver a entrenar el modelo.